In [148]:
import pandas as pd

In [150]:
# 1. Dosyayı yükle
df = pd.read_csv("hastane-sayisi-ozel.csv", sep=";", engine="python")

In [152]:
# 2. Yılları index yap (ilk sütun)
df = df.set_index(df.columns[0])

In [154]:
# 3. Transpoz: şehirler satıra, yıllar sütuna geçsin
df_transposed = df.T.reset_index()

In [156]:
# 4. 'index' sütununu 'il_adı' olarak yeniden adlandır
df_transposed = df_transposed.rename(columns={"index": "il_adı"})

In [158]:
# 5. Şehir adlarından "-1", "-2" gibi plaka kodlarını çıkar
df_transposed["il_adı"] = df_transposed["il_adı"].astype(str).str.replace(r"-\d+", "", regex=True)

In [160]:
# 6. Sayısal sütunları Int64 yap
for col in df_transposed.columns[1:]:
    df_transposed[col] = pd.to_numeric(df_transposed[col], errors="coerce").astype("Int64")

In [162]:
# 7. Temizlenmiş orijinal dosyayı kaydet
df_transposed.to_csv("hastane_yatak_bakanlik_duzenlenmis.csv", index=False, encoding="utf-8-sig")

In [164]:
# 8. Sırala - Türkçe olmayan düz alfabetik sıralama
df_sorted = df_transposed.sort_values(by="il_adı").reset_index(drop=True)
df_sorted.to_csv("hastane_yatak_bakanlik_duzenlenmis_alfabetik.csv", index=False, encoding="utf-8-sig")

In [166]:
# 9. Türk alfabesine göre sıralamak için harf karşılaştırma sözlüğü
turkce_alfabe = "AaBbCcÇçDdEeFfGgĞğHhIıİiJjKkLlMmNnOoÖöPpRrSsŞşTtUuÜüVvYyZz"
alfabe_sira = {harf: i for i, harf in enumerate(turkce_alfabe)}
def turkce_sirala(kelime):
    return [alfabe_sira.get(c, ord(c)) for c in kelime]

In [168]:
# 10. Türk alfabesine göre sıralama
df_sorted_tr = df_transposed.copy()
df_sorted_tr = df_sorted_tr.sort_values(by="il_adı", key=lambda x: x.map(turkce_sirala)).reset_index(drop=True)

In [170]:
# 11. Türkçe sıralı halini kaydet
df_sorted_tr.to_csv("hastane_yatak_bakanlik_duzenlenmis_alfabetik_turkce.csv", index=False, encoding="utf-8-sig")